# 🎯 Custom Fairness Constraints Tutorial

This tutorial shows how to define and implement custom fairness constraints for
synthetic data generation.

## Topics Covered

1. **Built-in Fairness Constraints**
2. **Creating Custom Constraints**
3. **Multi-Objective Optimization**
4. **Constraint Tuning and Validation**

## Setup

In [ ]:
# Imports
import sys
from pathlib import Path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.fairness.constraints import (
    BaseConstraint,
    DemographicParity,
    EqualizedOdds,
    ConstraintRegistry
)
from src.fairness.losses import FairnessLoss, MultiObjectiveLoss

import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")

## 1. Built-in Fairness Constraints

The framework provides several pre-built constraints. Use `ConstraintRegistry`
to list all available constraints.

In [ ]:
print("📋 Available Fairness Constraints:")
registry = ConstraintRegistry()
for name, constraint_class in registry.list_constraints().items():
    print(f"  {name}: {constraint_class}")

## 2. Creating Custom Constraints

Extend `BaseConstraint` and implement `compute_loss()` to define your own
fairness metric. Below is an example that enforces **representation parity**
(equal group sizes in generated data).

In [ ]:
class CustomRepresentationParity(BaseConstraint):
    """Ensure each sensitive group is equally represented."""

    def __init__(self, sensitive_attribute: str, tolerance: float = 0.05):
        super().__init__()
        self.sensitive_attribute = sensitive_attribute
        self.tolerance = tolerance

    def compute_loss(self, predictions, sensitive_values, **kwargs):
        unique, counts = torch.unique(sensitive_values, return_counts=True)
        ratios = counts.float() / sensitive_values.shape[0]
        target = torch.ones_like(ratios) / len(unique)
        deviation = torch.abs(ratios - target)
        return torch.clamp(deviation - self.tolerance, min=0).sum()

print("✅ Custom constraint ready")

## 3. Multi-Objective Optimization

Combine multiple constraints with different weights using `MultiObjectiveLoss`.

In [ ]:
constraints = {
    'dp': {'constraint': DemographicParity('gender', 0.1), 'weight': 1.0},
    'rp': {'constraint': CustomRepresentationParity('gender'), 'weight': 0.5}
}

multi_loss = MultiObjectiveLoss(constraints=constraints)
print("✅ Multi-objective setup ready")

## 4. Constraint Tuning

Sweep different thresholds to find the best fairness-fidelity trade-off.

In [ ]:
thresholds = [0.05, 0.1, 0.2]
for t in thresholds:
    print(f"Testing threshold {t}")